# Lab 2: API Client Integration

In this lab you will build a production-ready LLM API client using **LiteLLM** and **OpenRouter**.

**By the end of this lab you will know how to:**

- Securely load API keys from environment variables
- Make your first LLM API call via LiteLLM
- Use LiteLLM's built-in retry logic
- Measure Time To First Token (TTFT) for performance evaluation
- Use LiteLLM's built-in response caching
- Handle real-world API failures (timeouts, 5xx, malformed JSON)


## Step 1 — Environment Setup

We never hardcode API keys in source code. Instead, we load them from a `.env` file at runtime.

Create a `.env` file in this directory with:
```
OPENROUTER_API_KEY=sk-or-...
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # Reads .env from the current directory


def get_api_key() -> str:
    """Retrieve and validate the OpenRouter API key."""
    token = os.getenv("OPENROUTER_API_KEY")
    if not token:
        raise EnvironmentError(
            "OPENROUTER_API_KEY not found. "
            "Create a .env file with your key or set the environment variable."
        )
    return token


get_api_key()  # Validate early — fail fast if the key is missing
print("API key loaded successfully.")

## Step 2 — Your First API Call

[LiteLLM](https://docs.litellm.ai/docs/) provides a unified `completion()` interface across 100+ LLM providers.
We prefix the model name with `openrouter/` so LiteLLM knows to route through OpenRouter.

Key features:

- Direct Python library integration in your codebase
- Router with retry/fallback logic across multiple deployments (e.g. Azure/OpenAI) - Router
- Application-level load balancing and cost tracking
- Exception handling with OpenAI-compatible errors
- Observability callbacks (Lunary, MLflow, Langfuse, etc.)


In [ ]:
from litellm import completion

MODEL_ID = "openrouter/meta-llama/llama-3-8b-instruct:free"

prompt = "Explain what a vector database is in one paragraph."

response = completion(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=150,
    temperature=0.7,
)

print(response.choices[0].message.content)

## Step 3 — Retry Logic

Free-tier APIs are rate-limited. Instead of writing manual retry loops, LiteLLM has a built-in `num_retries` parameter that automatically retries on `RateLimitError` and network errors with exponential backoff.

In [ ]:
response = completion(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=150,
    temperature=0.7,
    num_retries=3,   # LiteLLM retries automatically on rate limit / network errors
    timeout=120,
)

print(response.choices[0].message.content)

## Step 4 — Measuring Time To First Token (TTFT)

Time To First Token (TTFT) is the most critical metric for perceived performance. Users perceive an application as fast if the first token arrives within 500ms.

To measure TTFT, we must enable streaming (`stream=True`) and record the time when the first chunk with content arrives.

In [ ]:
import time
from litellm import completion

MODEL_ID = "openrouter/meta-llama/llama-3-8b-instruct:free"

start = time.time()
first_token_time = None

response = completion(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "Tell me a long story about a brave knight."}],
    stream=True,
)

print("Streaming response:\n")
for chunk in response:
    if chunk.choices[0].delta.content:
        if first_token_time is None:
            first_token_time = time.time()
            ttft_ms = (first_token_time - start) * 1000
            print(f"\n\n>>> TTFT: {ttft_ms:.0f} ms <<<\n")
        print(chunk.choices[0].delta.content, end="", flush=True)


## Step 5 — Response Caching

Identical prompts can be expensive — and slow — if you re-call the model.
LiteLLM ships an in-memory cache that returns the previous response instantly on a cache hit.

**When to cache:**

- Repeated identical lookups (e.g., FAQ-style endpoints)
- Dev/test loops where you don't want to burn quota on the same prompt
- Batch jobs with deduplicated inputs

**When NOT to cache:**

- Chat UIs where you *want* variety per call
- Anything with `temperature > 0` and a user-visible randomness expectation


In [ ]:
import litellm
from litellm import completion
import time

# Enable in-memory caching. LiteLLM also supports Redis / S3 for shared production caches.
litellm.cache = litellm.Cache(type="local")

prompt = "Name three Saudi cities in one line."

# First call: cache MISS — actual API call
t0 = time.time()
r1 = completion(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    caching=True,
)
print(f"First call (miss):  {time.time() - t0:.2f}s")

# Second identical call: cache HIT — returns instantly
t0 = time.time()
r2 = completion(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    caching=True,
)
print(f"Second call (hit): {time.time() - t0:.2f}s")

assert r1.choices[0].message.content == r2.choices[0].message.content
print("\nCache returned the same content — zero API cost on the second call.")


## Step 6 — Handling Failures

Real API calls fail. Networks drop, providers 5xx, rate limits trip, and free-tier
models occasionally return malformed output. `num_retries` handles **transient**
errors; everything else needs explicit handling.

| Failure | Retry helps? | Why |
|---|---|---|
| 429 rate-limit | yes (with backoff) | Capacity issue, not your bug |
| 5xx server error | yes | Upstream blip |
| Network timeout | yes | Connection-level issue |
| JSON-parse error | **no** | The model returned text you can't use — retry won't help |
| 401 / invalid key | **no** | Configuration problem — fix it, don't retry |


In [ ]:
import json
from litellm import completion
from litellm.exceptions import (
    RateLimitError,
    Timeout,
    APIError,
    APIConnectionError,
    AuthenticationError,
)


def call_safely(prompt: str, expect_json: bool = False):
    """Call the model with structured error handling.

    Returns the response content (str or dict) on success, or None on a handled failure.
    """
    try:
        response = completion(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            num_retries=3,   # auto-retry on transient network / rate-limit / 5xx
            timeout=30,      # hard cap on total call duration
            max_tokens=200,
        )
    except AuthenticationError as e:
        # Don't retry — fix the key.
        print(f"Auth error — check your API key: {e}")
        return None
    except RateLimitError as e:
        print(f"Rate-limited even after retries: {e}")
        return None
    except Timeout as e:
        print(f"Call exceeded 30s budget: {e}")
        return None
    except (APIError, APIConnectionError) as e:
        # Provider 5xx, connection drop, etc.
        print(f"Upstream API error: {e}")
        return None

    content = response.choices[0].message.content

    if not expect_json:
        return content

    # JSON-parse failures are NOT retryable in the same loop — the model already
    # gave us text we can't use. Log and bail; caller decides whether to re-prompt.
    try:
        return json.loads(content)
    except json.JSONDecodeError as e:
        print(f"Model returned malformed JSON: {e}")
        print(f"Raw output: {content[:200]}")
        return None


# 1. Normal call — plain text response
print("--- plain text ---")
print(call_safely("Say hello in one word."))

# 2. JSON-parse call — free-tier models often slip on this. That's the point.
print("\n--- JSON parse ---")
print(call_safely(
    'Return a JSON object with keys "city" and "country" for Riyadh. '
    'JSON only, no prose, no markdown fence.',
    expect_json=True,
))
